# Conditioning and guidance

In the [training and sampling tutorial](./1.training_and_sampling.ipynb) we demonstrated how to sample from the learnt distribution **unconditionally**. 
In this tutorial we showcase the conditional sampling methods built-in `stix`. The code related to conditioning and guidance can be found in [`stix.sampling.guidance`](https://instadeepai.github.io/stix/api_reference/sampling/guidance.html) along with additional documentation. 

In order to easily visualize the effects of the different conditioning methods, we reuse the toy problem of previous tutorials: a 4-component Gaussian mixture with two modalities, `coordinates` (continuous, 2D) and `index` (discrete, the mixture component).

In `stix`, *conditional sampling* or *guidance* is implemented through *guidance recipes*. These are functions that handle the production of the per-modality [`Generator`](https://instadeepai.github.io/stix/api_reference/core/generator.html#stix.core.generator.Generator) from a given [`GenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#generative-models), optionally conditioning on some data. Guidance recipes must satisfy the [`GuidanceFn`](https://instadeepai.github.io/stix/api_reference/sampling/guidance.html#stix.sampling.guidance.GuidanceFn) protocol. The standard [`Solver`](https://instadeepai.github.io/stix/api_reference/sampling/solver.html#stix.sampling.solver.Solver) and [`ManualSolver`](https://instadeepai.github.io/stix/api_reference/sampling/solver_manual.html#stix.sampling.solver_manual.ManualSolver) configs accept guidance recipes through the `guidance_fn` argument (with an optional `guidance_scale`).

At sampling time, the guidance recipes modify the score produced by the [`GenerativeModel.get_generator`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.GenerativeModel.get_generator) based on some conditioning data.

We distinguish two types of conditioning data that can be passed to the model through two channels:

- <u>**Context Data :**</u> This data enriches the representation of standard modalities and are passed to the model's network forward. For instance, in an image generation model, a text prompt describing the desired output serves as context data.

- <u>**Intrinsic Data :**</u> This data corresponds to conditions on the interpolated variables and are not fed to the model's network. Methods that rely on this are referred to as *intrinsic guidance.* For example, in image generation, intrinsic data could be a specific region of the image that must remain fixed in the final output.

We will cover conditioning using both these channels and how to build your custom method throughout this tutorial. 
To keep this tutorial concise, we provide the user with pre-trained weights of the models used for sampling. These weights were obtained by training the same models on the corresponding tasks.

For further details on how to train a generative model, refer to the [1.training_and_sampling.ipynb](./1.training_and_sampling.ipynb) tutorial.

## Structure of the notebook

1. Imports
2. Setup: data, registry, and network
3. **Intrinsic guidance** 
    1. Unconditional sampling
    2. Intrinsic conditional sampling
    3. Conditioning on two intrinsic modalities
4. **Context conditioning** 
    1. Unconditional sampling
    2. Context conditional sampling
    3. Effect of the guidance weight
5. **Writing your own recipe** 
    1. The intrinsic and context targets
    2. A combined recipe
    3. Combined sampling

## 0. Installation

We recommend running this notebook in a **fresh virtual environment**.

Copy the notebook into some new directory. Then, from a terminal, in the new directory containing the notebook (`4.conditioning_and_guidance.ipynb`):
```
python -m venv my_env
source my_env/bin/activate
pip install notebook ipykernel

python -m ipykernel install --user --name my_env --display-name "my_env"

jupyter notebook
```

The next cell installs `stix` from PyPI together with the Hub client and plotting packages this notebook uses.

In [ ]:
%pip install stix-ml huggingface_hub matplotlib

## 1. Imports

The conditioning-specific pieces are the guidance recipes; everything else is the standard set-up from [1.training_and_sampling](./1.training_and_sampling.ipynb).

- [`stix.sampling.guidance`](https://instadeepai.github.io/stix/api_reference/sampling/guidance.html) : the **recipes** allow to leverage context and/or intrinsic data modality to obtain a conditional score.

In [ ]:
import logging
from functools import partial
from pathlib import Path

import grain
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
from flax import nnx
from huggingface_hub import snapshot_download
from jaxtyping import PyTree

from stix.core.embedder import IdentityEmbedder, OneHotDiscreteEmbedder
from stix.core.gen_model import GenerativeModel

# The generator type used at sampling time for continuous modalities.
from stix.core.generator import VelocityAndScore
from stix.core.interpolant import FlowMatchingOneSidedInterpolant
from stix.core.loss.criterion import CrossEntropyCriterion, MSECriterion
from stix.core.loss.utils import reduce_modality_losses
from stix.core.modality import Modality, ModalityRegistry
from stix.nn import (
    Network,
    NetworkDimsConfig,
    SumContextEncoder,
    TimeNoiseContextEncoder,
)

# The guidance recipes. Each one matches the `GuidanceFn` protocol and slots
# into a solver config via its `guidance_fn` field.
from stix.sampling.guidance import (
    get_classifier_free_guidance_generator,
    get_intrinsic_guidance_generator,
)

# The adaptive `diffrax` solver. Guidance is supplied through its config.
from stix.sampling.solver import Solver, SolverConfig

# Checkpointing: this notebook restores published weights instead of training,
# so the training loop of tutorial 1 never appears here (see Section 2.4).
from stix.training.checkpointer import Checkpointer, CheckpointerConfig
from stix.typing import Batch, Mask, RawSourceTargetPair, Var

In [ ]:
stix_logger = logging.getLogger("stix")
stix_logger.setLevel(logging.INFO)

key = jax.random.PRNGKey(0)

## 2. Set-up : DataLoader, ModalityRegistry, Network and Helpers

The different conditioning and guidance methods in this tutorial share some common boilerplate code that we detail below.

### 2.1 Data

The sampler is the one from tutorial 1. It generates 2d coordinates sampled from a gaussian mixture model as well as an index corresponding to the mixture component the coordinates were sampled from.

Like the samplers in the earlier notebooks, it is `jax.jit`-compiled and vectorised with `jax.vmap` over the batch of indices `grain` hands it, so one compiled call builds a whole batch.

Here we slightly modify the original code to add an `use_index_as_context` flag that allows to treat the label either as a standard data modality or use it as context. 
In the case, where it's used as context, it's not passed as a standard modality like in previous tutorial but rather stored in the `Batch` object under the `context_data` argument. 

We also include a `context_mask` that allows for context dropout. Context dropout randomly replaces the context data with a null token during training, which allows a single model to learn both conditional and unconditional generation so it can perform classifier-free guidance (which depends on both scores) during sampling.

In [ ]:
BATCH_SIZE = 256  # training batch size for gmm_dataset
CORNERS = jnp.array(
    [[-1.0, -1.0], [1.0, -1.0], [-1.0, 1.0], [1.0, 1.0]], dtype=jnp.float32
)  # the four modes of the Gaussian mixture toy problem
NUM_CLASSES = len(CORNERS)  # number of mixture components, i.e. the one-hot label size
# 0.5 is the value the published checkpoints were trained with. 0.2 left the
# null (context-free) branch too gradient-starved to infer the corner from
# `coordinates_t` alone: it plateaued at a 5x higher loss than the conditional
# branch and gave diffuse, uncommitted unconditional samples.
CONTEXT_DROPOUT_PROB = 0.5  # probability of dropping context per sample, used only when use_index_as_context=True

In [ ]:
# One compiled call builds the whole batch! See the pipeline below for more info.
# `use_index_as_context` is static: the two branches build differently-shaped
# `Batch` objects, so the choice has to be made at compile time.
@partial(jax.jit, static_argnames="use_index_as_context")
def sample_gmm(
    batch_indices: jax.Array, key: jax.Array, use_index_as_context: bool = False
) -> Batch:
    """Draw one GMM sample per index: pick a corner, scatter Gaussian noise around it.

    `grain` shuffles and groups the indices; we vectorise the draw over them with
    ``jax.vmap``, giving each index its own key via ``jax.random.fold_in`` so every
    sample is reproducible and `key` — one per stream — distinguishes train from val.

    Depending on the `use_index_as_context` flag, the corner label may be treated as
    context (similar to time t) or as a standard data modality.
    In the case where we include context, we implement simple context dropout
    to allow the network to predict the output without context, which is necessary
    for classifier-free guidance.
    """

    def draw_one(sample_key: jax.Array) -> tuple[jax.Array, jax.Array, jax.Array]:
        """One sample: the maths stays per-sample, ``vmap`` turns it into a batch."""
        idx_key, noise_key, dropout_key = jr.split(sample_key, 3)
        corner_index = jr.randint(idx_key, (), 0, NUM_CLASSES)
        coordinates = (
            jr.normal(noise_key, (2,), dtype=jnp.float32) * 0.2 + CORNERS[corner_index]
        )
        # Only consumed on the context branch; XLA drops it otherwise.
        keep_mask = (jr.uniform(dropout_key, ()) > CONTEXT_DROPOUT_PROB).astype(
            jnp.float32
        )
        return coordinates, corner_index, keep_mask

    keys = jax.vmap(partial(jr.fold_in, key))(batch_indices)
    coordinates, corner_index, keep_mask = jax.vmap(draw_one)(keys)
    corner_label = jax.nn.one_hot(corner_index, NUM_CLASSES)

    if not use_index_as_context:
        return Batch(
            raw_batch={
                "coordinates": RawSourceTargetPair(target=coordinates, source=None),
                "index": RawSourceTargetPair(target=corner_label, source=None),
            },
            is_discrete={"coordinates": False, "index": True},
        )

    return Batch(
        raw_batch={"coordinates": RawSourceTargetPair(target=coordinates, source=None)},
        is_discrete={"coordinates": False},
        context_data={"label": corner_label},
        context_mask={"label": keep_mask},
    )


def gmm_dataset(seed: int, use_index_as_context: bool = False):
    """Wire the batched ``Batch`` sampler into an infinite, batched grain dataset."""
    return (
        grain.MapDataset.range(int(1e9))
        .seed(seed)
        .shuffle()
        .repeat()
        .batch(BATCH_SIZE, drop_remainder=True)  # groups indices
        .map(  # indices -> batched Batch
            partial(
                sample_gmm,
                key=jr.key(seed),
                use_index_as_context=use_index_as_context,
            )
        )
        .to_iter_dataset()
    )

### 2.2 ModalityRegistry

Unchanged from tutorial 1: build the skeleton from a [``Batch``](https://instadeepai.github.io/stix/api_reference/typing/index.html#stix.typing.data.Batch) with [`from_batch`](https://instadeepai.github.io/stix/api_reference/core/modality.html#stix.core.modality.ModalityRegistry), then fill in the [``Interpolant``](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.Interpolant) and the per-modality [``Embedder``](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.Embedder)s. We wrap this in a function because we will need to build different registries for intrinsic and context conditioning.

In [ ]:
def build_registry(batch: Batch) -> ModalityRegistry:
    """Registry skeleton from a batch, plus a flow-matching interpolant and embedders."""
    # `from_batch` reads each modality's shape and discreteness; the remaining
    # fields are filled below.
    registry = ModalityRegistry.from_batch(batch)

    # Interpolant: the canonical flow-matching schedule (beta_t = t, gamma_t = 1 - t),
    # shared across every modality.
    registry.set("interpolant", FlowMatchingOneSidedInterpolant())

    # Embedders (shape-dependent, so installed with per-modality factories): a
    # one-hot embedder for discrete modalities, the identity embedder for continuous ones.
    registry.set(
        field="embedder",
        value=lambda modality: (
            OneHotDiscreteEmbedder(dm_shape=modality.shape)
            if modality.is_discrete
            else IdentityEmbedder(dm_shape=modality.shape)
        ),
        is_factory=True,
        use_deepcopy=True,
    )

    # Generator type: guided sampling operates on the score, so every modality
    # yields a `VelocityAndScore` generator.
    return registry

### 2.3 Network

We use a network very similar to the one used in the first tutorial. In this set-up, we add the possibility for the network to leverage `context_data` and `context_mask` (if provided) to add information to the standard data modalities (which are interpolated and passed under the `z_t` argument). For this, we use [``SumContextEncoder``](https://instadeepai.github.io/stix/api_reference/nn/context.html#stix.nn.SumContextEncoder) which is provided within the library. 

In [ ]:
def _is_module_leaf(leaf):
    """Stop pytree traversal at nnx.Module boundaries (the per-modality heads)."""
    return isinstance(leaf, nnx.Module)


class CrossModalMLPNetwork(Network):
    """Cross-modal MLP: concatenate every modality, fuse in a shared trunk, then
    route the fused state back through one head per modality.

    One class for both conditioning regimes, selected by `use_context_encoder`:

    - `False` (the default) builds the **conditioning-blind** network of tutorial 1:
    no context parameters at all, time is a raw scalar column, and passing
    `context_data` is an error rather than a silent no-op.
    - `True` adds a **context path**, delegated to `stix.nn.SumContextEncoder`:
    time and the one-hot label are each embedded, summed into one context
    vector, and concatenated into the trunk next to the modalities.
    """

    def __init__(
        self,
        embedding_dims: PyTree[int],
        hidden_dim: int,
        rngs: nnx.Rngs,
        use_context_encoder: bool = False,
        num_classes: int = NUM_CLASSES,
        context_dim: int = 32,
    ):
        """Shared trunk over the concatenated modalities (+ time, + context), plus one head each."""
        if use_context_encoder:
            self.context_dim = context_dim
            self.context_encoder = SumContextEncoder(
                time_encoder=TimeNoiseContextEncoder(
                    network_dims=NetworkDimsConfig(context_dim=context_dim),
                    gamma_fn=FlowMatchingOneSidedInterpolant().gamma_fn,
                    rngs=rngs,
                ),
                context_encoders={
                    "label": nnx.Sequential(
                        nnx.Linear(num_classes, context_dim, rngs=rngs),
                        nnx.silu,
                        nnx.Linear(context_dim, context_dim, rngs=rngs),
                    )
                },
            )
        else:
            self.context_dim = 0
            self.context_encoder = None

        fused_input_dim = sum(jax.tree.leaves(embedding_dims)) + (
            self.context_dim if self.context_encoder is not None else 1
        )
        self.trunk = nnx.Sequential(
            nnx.Linear(fused_input_dim, hidden_dim, rngs=rngs),
            nnx.silu,
            nnx.Linear(hidden_dim, hidden_dim, rngs=rngs),
            nnx.silu,
        )
        self.heads = nnx.data(
            jax.tree.map(
                lambda dim: nnx.Linear(hidden_dim, dim, rngs=rngs), embedding_dims
            )
        )

    def __call__(
        self, z_t, t, context_data=None, context_mask=None, attention_mask=None
    ):
        """Concatenate the modalities (+ time, + context); fuse; route back per modality."""
        modalities = jax.tree.leaves(z_t)
        batch_shape = modalities[0].shape[:-1]
        t_col = jnp.broadcast_to(jnp.atleast_1d(t), batch_shape + (1,))

        if self.context_encoder is None:
            if context_data is not None or context_mask is not None:
                raise ValueError(
                    "This network was built without a context path "
                    "(use_context_encoder=False), so it cannot consume context. Rebuild it "
                    "with use_context_encoder=True to condition through the context channel."
                )
            fused = self.trunk(jnp.concatenate(modalities + [t_col], axis=-1))
        else:
            context = self.context_encoder(t_col, context_data, context_mask)
            fused = self.trunk(jnp.concatenate(modalities + [context], axis=-1))

        return jax.tree.map(
            lambda head: head(fused), self.heads, is_leaf=_is_module_leaf
        )

### 2.4 Restore checkpoints helpers

We provide the following helpers to be able to restore our pre-trained models from our HuggingFace repo.

In [ ]:
HF_REPO_ID = "InstaDeepAI/STIX-tutorials"  # Hub repo hosting the published checkpoints
HF_CHECKPOINT_PREFIX = (
    "conditioning_and_guidance"  # subdirectory within HF_REPO_ID for this tutorial
)

In [ ]:
def download_checkpoint(name: str) -> str:
    """Fetch one published checkpoint directory, returning its local path."""
    remote = f"{HF_CHECKPOINT_PREFIX}/{name}"
    local_root = snapshot_download(HF_REPO_ID, allow_patterns=remote + "/*")
    return str(Path(local_root) / remote)


def restore_ema_model(gen_model, checkpoint_dir):
    """Load a checkpoint's debiased EMA weights into `gen_model`'s structure."""
    graphdef, params = nnx.split(gen_model, nnx.Param)
    checkpointer = Checkpointer(
        CheckpointerConfig(
            # Orbax rejects relative paths on restore, so resolve first.
            checkpoint_dir=str(Path(checkpoint_dir).resolve()),
            max_to_keep=None,  # read-only: never mutate a downloaded directory
        )
    )
    return nnx.merge(graphdef, checkpointer.restore_ema(params))

### 2.5 Plotting helpers

Throughout this notebook we will be plotting the results of our sampling, both conditional and unconditional. 
To center the focus of this tutorial on the part that matter (i.e. the conditioning), we provide some boilerplate code for plotting our samples. 

In [ ]:
def plot_samples(ax, samples, title, reference=None):
    """Scatter generated coordinates, coloured by predicted mode."""
    coords = samples["coordinates"]
    # Discrete modalities decode to label indices, not one-hot rows.
    predicted_modes = samples["index"]
    if reference is not None:
        ax.scatter(reference[:, 0], reference[:, 1], s=6, c="lightgrey", zorder=0)
    ax.scatter(
        coords[:, 0],
        coords[:, 1],
        s=7,
        c=predicted_modes,
        cmap=plt.get_cmap("tab10", NUM_CLASSES),
        vmin=-0.5,
        vmax=NUM_CLASSES - 0.5,
        alpha=0.6,
    )

    ax.set_title(title, fontsize=10)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-2.5, 2.5)
    ax.set_aspect("equal")

In [ ]:
def plot_context_samples(ax, samples, title, reference=None):
    """Scatter generated coordinates (this model has no `index` output to colour by)."""
    coords = samples["coordinates"]
    if reference is not None:
        ax.scatter(reference[:, 0], reference[:, 1], s=6, c="lightgrey", zorder=0)
    ax.scatter(coords[:, 0], coords[:, 1], s=7, c="tab:blue", alpha=0.6)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-2.5, 2.5)
    ax.set_aspect("equal")

## 3. Intrinsic guidance

We discuss a first guidance recipe, which we call *intrinsic guidance*, that leverage data specifying conditions on the generated data modalities values. This guidance recipe modifies the score following the well-known *classifier guidance* approach. Writing $c$ for the conditioning data, and $s_\theta(z_t, t)$ for **the trained network's own score estimate**, Bayes' rule gives

$$\underbrace{\nabla_{z_t} \log p_t(z_t \mid c)}_{\text{conditional score}} = \underbrace{\nabla_{z_t} \log p_t(z_t)}_{\approx\ s_\theta(z_t, t)\ \text{(the model we trained)}} + \underbrace{\nabla_{z_t} \log p_t(c \mid z_t)}_{\text{needs a classifier}}.$$

In a standard set-up, we would use an externally trained model to be to obtain $p_t(c \mid z_t)$. A simple alternative is to leverage the model's own prediction and build a surrogate conditional score by introducing a guidance loss $\ell(z_t, c)$ and differentiating it w.r.t. $z_t$.

$$\nabla_{z_t} \log p_t(z_t \mid c) \simeq -\nabla_{z_t}\ell(z_t, c).$$

Having defined this conditional score, 

$$s_{\text{cond}}(z_t, t) = s_\theta(z_t, t) - G(t) \, \nabla_{z_t} \ell(z_t, c),$$

with $G(t)$ a per-modality **guidance scale** we choose at sampling time.

The method we just described is implemented in the
[`get_intrinsic_guidance_generator`](https://instadeepai.github.io/stix/api_reference/sampling/guidance.html#stix.sampling.guidance.get_intrinsic_guidance_generator) recipe. This specific recipe requires to defines the guidance loss $\ell$. This is done by overriding the [`get_guidance_loss`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.GenerativeModel.get_guidance_loss) method of the considered [`GenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.GenerativeModel). In general, the exact form of the guidance loss depends on the interpretation of the model's network output. For this reason, we do not provide a default working implementation of the [`get_guidance_loss`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.GenerativeModel.get_guidance_loss), and let the user implement it consistently with the specificities of their own model.

> **Note**: Remark that the guidance recipes always have access to [`GenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.GenerativeModel) and its methods, which offers additional flexibilty when desining your own recipe.

To go further, we thus need to specify a generative model. In the following we use the `HybridVelocityLogitsGenerativeModel` introduced at the end of tutorial [3.generative_model.ipynb](./3.generative_model.ipynb). Recall that this model predicts the velocity for the continuous modalities and the logits for discrete ones. With this model, a natural choice of the guidance loss is to use:

- A cross-entropy between the logits and the raw target variables for discrete modalities,
- An MSE on the velocity for continuous modalities.

> **Note**: One could have used a model predicting probabilites for the target variable instead of logits, and extract logits from predicted probabilities using a log. However, it is numerically more stable to directly predict the logits and using them to compute the cross-entropy.

In [ ]:
intrinsic_train_iterator = iter(gmm_dataset(seed=0, use_index_as_context=False))
intrinsic_validation_iterator = iter(gmm_dataset(seed=42, use_index_as_context=False))

batch = next(intrinsic_train_iterator)
intrinsic_modality_registry = build_registry(batch)

In [ ]:
# One weight for continuous datamodes, another for discrete.
# Upweight the discrete modalities by 5.
loss_weights = intrinsic_modality_registry.map(
    lambda modality: 5 if modality.is_discrete else 1.0
)


class HybridVelocityLogitsGenerativeModel(GenerativeModel):
    """Hybrid velocity (continuous modalities) and logits (discrete) generative model."""

    def get_loss(self, net_out, t, z_t, epsilon, embedded_pairs, raw_pairs, loss_mask):
        """Mixed loss: velocity MSE for continuous modalities, cross-entropy for discrete ones."""
        # A criterion for each kind of modality.
        mse = MSECriterion()
        cross_entropy = CrossEntropyCriterion()

        # Branch on `modality.is_discrete` to pick the right target and criterion.
        def _per_modality_network_output_to_loss(
            modality, net_out_k, embedded_pairs_k, raw_pairs_k, epsilon_k, loss_mask_k
        ):
            # Discrete
            if modality.is_discrete:
                # cross_entropy takes logits as input, so net_out_k is interpreted as logits
                return cross_entropy(net_out_k, raw_pairs_k.target, t, loss_mask_k)

            # Continuous
            conditional_velocity = modality.interpolant.get_conditional_velocity(
                embedded_pairs_k, t, epsilon_k
            )
            return mse(net_out_k, conditional_velocity, t, loss_mask_k)

        # Use `.map` to apply the function over the whole pytree
        per_modality_loss = self.modality_registry.map(
            _per_modality_network_output_to_loss,
            net_out,
            embedded_pairs,
            raw_pairs,
            epsilon,
            loss_mask,
        )

        # Reduce the pytree to a scalar by computing the mean
        return reduce_modality_losses(per_modality_loss, weights=loss_weights)

    def get_generator(self, net_out, z_t, t):
        r"""Convert network output to a per-modality `VelocityAndScore` generator.

        Continuous: the output already *is* the velocity (identity); the score is
        derived from it via `score_from_velocity`.
        Discrete: the flow lives in embedding space. `index` uses a one-hot
        embedder, so the embedding space is the simplex and the expected embedding
        under the predicted categorical is just the distribution itself:
        z1_hat = softmax(logits). Then `velocity_from_target` / `score_from_target`.
        """

        def _per_modality_network_output_to_generator(modality, net_out_k, z_t_k):
            # Discrete
            if modality.is_discrete:
                z1_hat = jax.nn.softmax(net_out_k, axis=-1)
                velocity = modality.interpolant.velocity_from_target(z1_hat, z_t_k, t)
                score = modality.interpolant.score_from_target(z1_hat, z_t_k, t)
                return VelocityAndScore(velocity=velocity, score=score)
            # Continuous
            velocity = net_out_k
            score = modality.interpolant.score_from_velocity(net_out_k, z_t_k, t)
            return VelocityAndScore(velocity=velocity, score=score)

        return self.modality_registry.map(
            _per_modality_network_output_to_generator, net_out, z_t
        )

    def get_guidance_loss(
        self,
        net_out,
        z_t,
        t,
        intrinsic_data,
        intrinsic_mask,
    ):
        """Implement the intrinsic guidance loss, using a cross-entropy
        on logits and one-hot encoded targets for the discrete modalities
        and an MSE on the predicted velocity for the continuous ones.
        """
        # A whole ``None`` mask means "fully conditional" (no masking) across every
        # modality the pytree analogue of an all-ones mask, and the default.
        # Synthesize an all-``None`` mask matching ``intrinsic_data`` so it is
        # treated identically to a per-leaf ``None`` (a bare ``None`` would otherwise
        # trip the structure check below). Target-shaped, so it stays pytree-generic.
        if intrinsic_mask is None:
            intrinsic_mask = jax.tree.map(lambda _: None, intrinsic_data)

        # Readable error if targets and masks disagree in structure; the map below
        # would otherwise fail deep inside jax.
        if jax.tree.structure(intrinsic_data) != jax.tree.structure(
            intrinsic_mask, is_leaf=lambda x: x is None
        ):
            raise ValueError(
                "intrinsic_data and intrinsic_mask must have the same structure."
            )

        # The intrinsic channel must span exactly the registry modalities — the
        # caller supplies it already restricted to registry keys. Fail loud here on
        # any missing/extra key rather than deep inside the per-modality map.
        if jax.tree.structure(intrinsic_data) != self.modality_registry.treedef:
            raise ValueError(
                "intrinsic_data must match the model's modality registry structure "
                "(one full-shape target per registry modality)."
            )

        # Loss fns require a real mask, so fill any None entries with all-ones
        # broadcast to intrinsic_data's shape. intrinsic_data must be the first
        # jax.tree.map argument since intrinsic_mask may contain None (which map skips).
        filled_intrinsic_mask = jax.tree.map(
            lambda intrinsic_data_k, intrinsic_mask_k: (
                intrinsic_mask_k
                if intrinsic_mask_k is not None
                else jnp.ones_like(intrinsic_data_k)
            ),
            intrinsic_data,
            intrinsic_mask,
        )

        # Turn this model's own prediction into the per-modality quantity each
        # criterion scores against the condition: for discrete modalities the native
        # logits this head predicts (cross-entropy), for continuous ones the clean
        # target recovered from the predicted velocity (MSE).

        def _compute_loss_per_modality(
            modality: Modality,
            net_out_k: Var,
            z_t_k: Var,
            intrinsic_data_k: Var,
            intrinsic_mask_k: Mask,
        ):
            if modality.is_discrete:
                return CrossEntropyCriterion()(
                    net_out_k, intrinsic_data_k, t, intrinsic_mask_k
                )
            else:
                estimated_z1 = modality.interpolant.target_from_velocity(
                    net_out_k, z_t_k, t
                )
                return MSECriterion()(
                    estimated_z1, intrinsic_data_k, t, intrinsic_mask_k
                )

        losses = self.modality_registry.map(
            _compute_loss_per_modality,
            net_out,
            z_t,
            intrinsic_data,
            filled_intrinsic_mask,
        )

        return jax.tree.reduce(jnp.add, losses, jnp.zeros(()))

We then instantiate the generative model before restoring the pretrained weights using the helpers defined in section [2.4](#24-restore-checkpoints-helpers).

In [ ]:
key, joint_model_key = jr.split(key)
joint_model = HybridVelocityLogitsGenerativeModel(
    network=CrossModalMLPNetwork(
        embedding_dims=intrinsic_modality_registry.map(
            lambda modality: modality.embedder.embedding_shape[-1]
        ),
        hidden_dim=128,
        rngs=nnx.Rngs(joint_model_key),
    ),
    modality_registry=intrinsic_modality_registry,
)

# The freshly-initialised model above supplies the structure; the checkpoint
# supplies the weights.
joint_model = restore_ema_model(joint_model, download_checkpoint("joint_model"))

In [ ]:
NUM_SAMPLES = 512  # number of samples drawn in every sampling demo
MAX_SOLVER_STEPS = 1000  # max adaptive solver steps per sampling trajectory
STOCHASTICITY_SCALE = 1.0  # scales gamma_t in the SDE diffusion term

# The function defined here, setting it proportional to \gamma(t), corrects
# for unfavourable asymptotic behavior of the interpolant of choice.
intrinsic_stochasticity_scale_fn = intrinsic_modality_registry.map(
    lambda modality: (lambda t: STOCHASTICITY_SCALE * modality.interpolant.gamma_fn(t))
)

# Define shared set of keys between the 2 solvers for comparison.
key, key_init, key_solve = jr.split(key, 3)
solve_keys = jr.split(key_solve, NUM_SAMPLES)

# Instantiate initial state for both unconditional and conditional solvers
z_init = intrinsic_modality_registry.sample_initial_state(
    key_init, num_samples=NUM_SAMPLES
)

### 3.1 Unconditional Sampling

Below, we quickly showcase how we perform unconditional sampling. In this case, we obtain unconditional samples by using a non-guided solver with the same trained model we use for intrinsic conditioning. 
Indeed, the model is not trained with a specific conditioning recipe and can thus be used for unconditional sampling as well. 


We used the unconditional samples to compare and visualise the effects of conditioning and guidance.

In [ ]:
# To be able to compare the effect of conditioning and guidance,
# we first draw a batch of unconditional samples from the joint model.
unconditional_solver = Solver(
    SolverConfig(
        stochasticity_scale=intrinsic_stochasticity_scale_fn,
        max_steps=MAX_SOLVER_STEPS,
    )
)

unconditional_samples = jax.vmap(lambda x, k: unconditional_solver(joint_model, x, k))(
    z_init, solve_keys
)

### 3.2 Intrinsic conditional sampling

We provide the intrinsic data below. For our example, with adopt the following rules : 

1. `intrinsic_data` spans **exactly** the registry modalities — one entry per modality, no more, no fewer.
2. Each entry is **full shape**, in the space the criterion compares against: the **raw** (de-embedded) space for continuous modalities, the **one-hot** label for discrete ones.
3. Partial conditioning is expressed by `intrinsic_mask`, *not* by omitting entries. Supply filler for the parts you do not condition on and mask them out. A `None` mask (whole-tree or per-leaf) means "fully conditional".

So conditioning on the label alone means: a real one-hot for `index` with an all-ones mask, and zero-filler for `coordinates` with an all-zeros mask.

In [ ]:
TARGET_LABEL = 3  # the corner at (+1, +1); the label we condition on throughout

In [ ]:
label_intrinsic_data = {
    "coordinates": jnp.zeros((2,)),  # filler, masked out below
    "index": jax.nn.one_hot(TARGET_LABEL, NUM_CLASSES),
}
label_intrinsic_mask = {
    "coordinates": jnp.zeros((2,)),  # 0 -> do not condition on the coordinates
    "index": jnp.ones((NUM_CLASSES,)),  # 1 -> condition on the label
}

In [ ]:
INTRINSIC_GUIDANCE_WEIGHT = 30
# The function defined here, setting it proportional to \beta(t), corrects
# for unfavourable asymptotic behavior of the interpolant of choice.
# More precisely, it enforces the conditional score to be zero at t=0.
# Indeed at t=0, the conditional score can be very large and push the sample out of distribution.
# The solver then struggles to bring them back to a sensible point in the sampling space.

# The scaling value of 30 is arbitrary, in section 4.3 we study the effect
# of the guidance weight when performing context conditioning.
intrinsic_guidance_scale_fn = intrinsic_modality_registry.map(
    lambda modality: (
        lambda t: INTRINSIC_GUIDANCE_WEIGHT * modality.interpolant.beta_fn(t)
    )
)

# Same diffrax `Solver` as above, configured with two extra fields: a guidance
# recipe and a per-modality guidance scale.
intrinsic_guided_solver = Solver(
    SolverConfig(
        stochasticity_scale=intrinsic_stochasticity_scale_fn,
        max_steps=MAX_SOLVER_STEPS,
        guidance_fn=get_intrinsic_guidance_generator,
        guidance_scale=intrinsic_guidance_scale_fn,
    )
)

intrinsic_samples = jax.vmap(
    lambda x, k: intrinsic_guided_solver(
        joint_model,
        x,
        k,
        intrinsic_data=label_intrinsic_data,
        intrinsic_mask=label_intrinsic_mask,
    )
)(z_init, solve_keys)

In [ ]:
reference_coords = next(intrinsic_validation_iterator).raw_batch["coordinates"].target
fig, axes = plt.subplots(1, 2, figsize=(8.4, 4.2))
plot_samples(
    axes[0], unconditional_samples, "Unconditional", reference=reference_coords
)
plot_samples(
    axes[1],
    intrinsic_samples,
    f"Intrinsic guidance on index={TARGET_LABEL}",
    reference=reference_coords,
)
plt.tight_layout()
plt.show()

### 3.3 Conditioning on two intrinsic modalities

Because all the modalities we trained on are available at sampling time for intrinsic guidance, we can do more advanced conditioning by combining multiple modalities together. 
Here we choose to condition on

- `coordinates` target $(+1, \star)$ with mask $(1, 0)$ — the $y$ entry is filler;
- `index` condition on top right corner (index 3)

In [ ]:
TARGET_X = 1.0  # the x coordinate we pin via intrinsic guidance throughout

In [ ]:
combined_intrinsic_data = {
    "coordinates": jnp.array([TARGET_X, 0.0]),  # second entry is filler
    "index": jax.nn.one_hot(TARGET_LABEL, NUM_CLASSES),
}
combined_intrinsic_mask = {
    "coordinates": jnp.array([1.0, 0.0]),  # condition on x only
    "index": jnp.ones((NUM_CLASSES,)),  # 1 -> condition on the label
}

In [ ]:
fig, ax = plt.subplots(figsize=(4.4, 4.2))

x_samples = jax.vmap(
    lambda x, k: intrinsic_guided_solver(
        joint_model,
        x,
        k,
        intrinsic_data=combined_intrinsic_data,
        intrinsic_mask=combined_intrinsic_mask,
    )
)(z_init, solve_keys)

coords = x_samples["coordinates"]

plot_samples(
    ax,
    x_samples,
    title=None,
    reference=reference_coords,
)
ax.axvline(1.0, ls="--", c="crimson", lw=1)
fig.suptitle("Conditioning on x = +1 and label = 3; y is free")
plt.tight_layout()
plt.show()

# 4. Context Conditioning

Let us now discuss a guidance recipe using `context_data` that are fed to the model's network. This recipe follows the *classifier-free guidance* method. Like previously, we have with guidance scale $G(t)$ :

$$      s_{\text{cond}}(z_t, t)
        = \underbrace{\nabla_{z_t} \log p_t(z_t)}_{\text{unconditional score}}
        + G(t) \underbrace{\nabla_{z_t} \log p_t(c | z_t)}_{\text{guidance gradient}}.
$$

Where,

$$
        \nabla_{z_t} \log p_t(z_t)
        \approx \underbrace{s_{\theta}(z_t, t, \text{context}=\text{null})}_{\text{unconditional score}},
$$

and,

$$        
        \nabla_{z_t} \log p_t(c | z_t)
        \approx \underbrace{s_{\theta}(z_t, t, \text{context}=c)}_{\text{conditional score}}
        - \underbrace{s_{\theta}(z_t, t, \text{context}=\text{null})}_{\text{unconditional score}},
$$

thus

$$
        s_{\text{cond}}(z_t, t)
            \approx G(t) s_{\theta}(z_t, t, \text{context}=c)
            + (1 - G(t)) s_{\theta}(z_t, t, \text{context}=\text{null}).
$$

With this method we need two forward passes per integration step : one with context set to `null` and one with the conditioning context. 
We notice that setting $G(t)=1$ recovers simple conditional sampling while setting $G(t)=0$ recovers unconditional sampling. An informed reader would notice that $s_{\theta}(z_t, t, \text{context}=c)$ targets $\nabla_{z_t} \log p(z_t|c)$ directly but it is commonly known in the literature that amplifying the conditioning signal can lead to improved results.

In [ ]:
context_train_iterator = iter(gmm_dataset(seed=0, use_index_as_context=True))
context_validation_iterator = iter(gmm_dataset(seed=42, use_index_as_context=True))

batch = next(context_train_iterator)
context_modality_registry = build_registry(batch)

We then instantiate the generative model before restoring the pretrained weights using the helpers defined in section [2.4](#24-restore-checkpoints-helpers).

It's important to note that in this case, our model doesn't generate the labels. At sampling time, it only generates 2D-coordinates as the labels are only used as conditioning info and are not learned as part of a joint distribution. 

In [ ]:
key, context_model_key = jr.split(key)
context_model = HybridVelocityLogitsGenerativeModel(
    network=CrossModalMLPNetwork(
        embedding_dims=context_modality_registry.map(
            lambda modality: modality.embedder.embedding_shape[-1]
        ),
        hidden_dim=128,
        rngs=nnx.Rngs(context_model_key),
        use_context_encoder=True,
        num_classes=NUM_CLASSES,
    ),
    modality_registry=context_modality_registry,
)

# The freshly-initialised model above supplies the structure; the checkpoint
# supplies the weights.
context_model = restore_ema_model(context_model, download_checkpoint("context_model"))

In [ ]:
# Similarly to section 3, we define the necessary elements to run our solvers.
context_stochasticity_scale_fn = context_modality_registry.map(
    lambda modality: (lambda t: STOCHASTICITY_SCALE * modality.interpolant.gamma_fn(t))
)

key, key_init, key_solve = jr.split(key, 3)
solve_keys = jr.split(key_solve, NUM_SAMPLES)

z_init = context_modality_registry.sample_initial_state(
    key_init, num_samples=NUM_SAMPLES
)

### 4.1 Unconditional Sampling

Below, we quickly showcase how we perform unconditional sampling. In this case, we obtain unconditional samples by using a non-guided solver and enforcing `context_data=None`. Indeed, during training we use a context dropout to ensure the trained model is also able to sample unconditionally. 


We used the unconditional samples to compare and visualise the effects of conditioning and guidance.

In [ ]:
# To be able to compare the effect of conditioning and guidance,
# we first draw a batch of unconditional samples from the context model,
# obtained by passing context_data=None (the null branch context dropout trained).
unconditional_solver = Solver(
    SolverConfig(
        stochasticity_scale=context_stochasticity_scale_fn,
        max_steps=MAX_SOLVER_STEPS,
    )
)

unconditional_samples = jax.vmap(
    lambda x, k: unconditional_solver(
        context_model, x, k, context_data=None, context_mask=None
    )
)(z_init, solve_keys)

### 4.2 Context conditional sampling

In [ ]:
target_context = {"label": jax.nn.one_hot(TARGET_LABEL, NUM_CLASSES)}

In [ ]:
CONTEXT_GUIDANCE_WEIGHT = 1.0
context_guidance_scale_fn = context_modality_registry.map(
    lambda modality: (lambda t: CONTEXT_GUIDANCE_WEIGHT)
)

context_guided_solver = Solver(
    SolverConfig(
        stochasticity_scale=context_stochasticity_scale_fn,
        max_steps=MAX_SOLVER_STEPS,
        guidance_fn=get_classifier_free_guidance_generator,
        guidance_scale=context_guidance_scale_fn,
    )
)

context_conditional_samples = jax.vmap(
    lambda x, k: context_guided_solver(
        context_model, x, k, context_data=target_context, context_mask=None
    )
)(z_init, solve_keys)

In [ ]:
context_reference_coords = (
    next(context_validation_iterator).raw_batch["coordinates"].target
)
fig, axes = plt.subplots(1, 2, figsize=(8.4, 4.2))
plot_context_samples(
    axes[0],
    unconditional_samples,
    "Unconditional (context=None)",
    reference=context_reference_coords,
)
plot_context_samples(
    axes[1],
    context_conditional_samples,
    f"Context-conditional on label={TARGET_LABEL}",
    reference=context_reference_coords,
)
plt.tight_layout()
plt.show()

Performance of unconditional can seem a bit low, especially when compared with section [3.1](#31-unconditional-sampling). However, here the model is only leveraging a single data modality. This low performance is also due to the minimalistic network we're using for this tutorial. 

### 4.3 Effect of the guidance weight

> The guidance weight is not equivalent between intrinsic and context conditioning. However the expected effect is similar. To keep this tutorial concise, we only showcase the effects of different values of the guidance weight for context guidance. 

In [ ]:
SWEEP_GUIDANCE_WEIGHTS = [0.0, 1.0, 4.0, 10.0]

fig, axes = plt.subplots(
    1, len(SWEEP_GUIDANCE_WEIGHTS), figsize=(4.4 * len(SWEEP_GUIDANCE_WEIGHTS), 4.2)
)
for ax, weight in zip(axes, SWEEP_GUIDANCE_WEIGHTS):
    weight_guided_solver = Solver(
        SolverConfig(
            stochasticity_scale=context_stochasticity_scale_fn,
            max_steps=MAX_SOLVER_STEPS,
            guidance_fn=get_classifier_free_guidance_generator,
            guidance_scale=context_modality_registry.map(
                lambda modality: (lambda t: jnp.asarray(weight, dtype=jnp.float32))
            ),
        )
    )

    weight_samples = jax.vmap(
        lambda x, k: weight_guided_solver(
            context_model, x, k, context_data=target_context, context_mask=None
        )
    )(z_init, solve_keys)

    plot_context_samples(
        ax,
        weight_samples,
        f"G = {weight:g}",
        reference=context_reference_coords,
    )
fig.suptitle(f"Classifier-free guidance toward label={TARGET_LABEL}, sweeping G")
plt.tight_layout()
plt.show()

Section 4.2 fixed $G(t)$ to an arbitrary constant (1.0). Recall from Section 4's derivation that $G$ linearly interpolates between the unconditional score ($G = 0$) and the context-conditional score ($G = 1$), and strengthen the conditional score for $G > 1$. Sweeping it shows concentration around the target corner improving as $G$ grows past 1, but with sharply diminishing returns: most of the gain happens between $G = 0$ and $G = 1$, and pushing $G$ far past 1 buys comparatively little extra. Push $G$ far enough and the samples end up *further* from the target than the unconditional ones, not closer.

We reuse the same initial noise (`z_init`, `solve_keys`) for every $G$ below, so any difference between panels is attributable to the guidance weight alone, and report the mean distance to the target corner as a simple fidelity proxy.

# 5. Writing your own recipe


To define a custom guidance recipe, it suffices to implement a function satisfying the [`GuidanceFn`](https://instadeepai.github.io/stix/api_reference/sampling/guidance.html) protocol. In 
sections 3 and 4 we showed how to use already implemented recipes, each of which leveraged only one of the two available conditioning data channels. In this section, we merge these two recipes into a single custom one.

Combining the scores gives, up to terms constant in $z_t$:

$$
\begin{aligned}
s_{\text{cond}}(z_t, t)
    &= \underbrace{s_{\theta}(z_t, t, \text{context}=\text{null})}_{\text{unconditional score}} \\
    &\quad + G_{\text{cfg}}(t) \Big(\underbrace{s_{\theta}(z_t, t, \text{context}=c)}_{\text{conditional score}} - s_{\theta}(z_t, t, \text{context}=\text{null})\Big) \\
    &\quad - G_{\text{intr}}(t) \, \nabla_{z_t} \mathcal{L}(\hat{z}_{\mathrm{tgt}}(z_t), x).
\end{aligned}
$$

The first line is Section 4's classifier free guidance term unchanged; the last term is Section 3's intrinsic correction, differentiated through the same null context forward rather than through a forward that never saw context at all. Since the two scales play different roles, we bake them into the recipe itself instead of the single `guidance_scale` field `SolverConfig` expects.

### 5.1 The intrinsic and context targets

We reuse Section 3.3's partial conditioning on the x coordinate, pinning x = 1 and leaving y free. The intrinsic channel contract from Section 3.1 still applies, but `context_modality_registry` only contains `coordinates` (Section 4 moved the label out into context), so unlike Section 3.3's `combined_intrinsic_data` there is no `index` entry to supply.

In [ ]:
# Context target
target_context = {"label": jax.nn.one_hot(TARGET_LABEL, NUM_CLASSES)}

# Intrinsic target
x_intrinsic_data = {"coordinates": jnp.array([TARGET_X, 0.0])}  # second entry is filler
x_intrinsic_mask = {"coordinates": jnp.array([1.0, 0.0])}  # condition on x only

### 5.2 A combined recipe

`SolverConfig` accepts a single `guidance_scale`, enough for any one recipe in `stix.sampling.guidance` but not for two scales playing different roles at once. We write a factory that closes over both scales and returns a plain function matching the `GuidanceFn` protocol.

In [ ]:
def combine_generators(
    modality,
    generator_uncond,
    generator_context,
    grad,
    z_t_k,
    intrinsic_scale,
    cfg_scale,
    t,
):
    """Build a `VelocityAndScore` from the combined conditional score.

    s_cond = uncond + G_cfg(t) * (context - uncond) - G_intr(t) * grad.

    The middle term alone is exactly `get_classifier_free_guidance_generator`;
    the last term alone is exactly `get_intrinsic_guidance_generator`'s
    correction. We read each branch's score off its generator, combine, then
    re-derive a consistent velocity via the interpolant's `velocity_from_score`
    (exactly what the library recipes do internally).
    """
    score_uncond = generator_uncond.score
    score_context = generator_context.score
    cfg_term = cfg_scale(t) * (score_context - score_uncond)
    intrinsic_term = intrinsic_scale(t) * grad
    conditional_score = score_uncond + cfg_term - intrinsic_term
    velocity = modality.interpolant.velocity_from_score(conditional_score, z_t_k, t)
    return VelocityAndScore(velocity=velocity, score=conditional_score)


def make_combined_guidance_fn(intrinsic_scale_fn, cfg_scale_fn):
    """Build a GuidanceFn combining classifier free guidance with intrinsic guidance.

    This composes the two library recipes directly rather than calling either
    one: the context term below is exactly what `get_classifier_free_guidance_generator`
    computes, and the intrinsic term is exactly what `get_intrinsic_guidance_generator`
    computes. Both scales are closed over rather than threaded through the
    solver's single `guidance_scale` field, following the note in `SolverConfig`.

    Calling those two recipes separately would each redo the null context
    forward. Here it is done once and shared: `compute_null_context_loss`
    below feeds both the unconditional generator and, through `jax.value_and_grad`,
    the intrinsic loss gradient, exactly as `get_intrinsic_guidance_generator` does
    internally. Like every recipe, it returns a per-modality `Generator`.
    """

    def combined_guidance_fn(
        gen_model,
        z_t,
        t,
        context_data=None,
        context_mask=None,
        intrinsic_data=None,
        intrinsic_mask=None,
        *,
        attention_mask=None,
        guidance_scale=None,
    ):
        def compute_null_context_loss(z_t):
            """Intrinsic loss from the null context forward (context_data=None).

            Differentiating this w.r.t. `z_t` gives the intrinsic loss
            gradient below; its `net_out_null` aux also gives the unconditional
            generator, so this one forward pass covers both, avoiding a second one.
            """
            net_out_null = gen_model.get_network_output(
                z_t, t, None, None, attention_mask
            )
            loss = gen_model.get_guidance_loss(
                net_out_null, z_t, t, intrinsic_data, intrinsic_mask
            )
            return loss, net_out_null

        (_, net_out_null), grad_intrinsic_loss = jax.value_and_grad(
            compute_null_context_loss, has_aux=True
        )(z_t)
        generator_unconditional = gen_model.get_generator(net_out_null, z_t, t)

        net_out_context = gen_model.get_network_output(
            z_t, t, context_data, context_mask, attention_mask
        )
        generator_context = gen_model.get_generator(net_out_context, z_t, t)

        # `t` is a plain scalar, not a per-modality tree, so it can't be one of
        # `.map`'s zipped tree arguments (unlike the six below).
        return gen_model.modality_registry.map(
            partial(combine_generators, t=t),
            generator_unconditional,
            generator_context,
            grad_intrinsic_loss,
            z_t,
            intrinsic_scale_fn,
            cfg_scale_fn,
        )

    return combined_guidance_fn

In [ ]:
# The weights are adjusted here to balance the two guidance terms
INTRINSIC_WEIGHT = 20.0
CONTEXT_WEIGHT = 0.75

intrinsic_scale_fn = context_modality_registry.map(
    lambda modality: (lambda t: INTRINSIC_WEIGHT)
)
cfg_scale_fn = context_modality_registry.map(
    lambda modality: (lambda t: CONTEXT_WEIGHT)
)
combined_stochasticity_scale_fn = context_modality_registry.map(
    lambda modality: (lambda t: STOCHASTICITY_SCALE * modality.interpolant.gamma_fn(t))
)

combined_solver = Solver(
    SolverConfig(
        stochasticity_scale=combined_stochasticity_scale_fn,
        max_steps=MAX_SOLVER_STEPS,
        guidance_fn=make_combined_guidance_fn(intrinsic_scale_fn, cfg_scale_fn),
    )
)

### 5.3 Combined sampling

We sample from `context_model` with both channels active: `intrinsic_data` pins x = 1 the same way Section 3.3 did, `context_data` steers the label the same way Section 4.3 did. `context_model` never learned a joint distribution over the label, so the plot below reuses `plot_context_samples`; the crimson line marks the pinned x.

In [ ]:
combined_samples = jax.vmap(
    lambda x, k: combined_solver(
        context_model,
        x,
        k,
        context_data=target_context,
        context_mask=None,
        intrinsic_data=x_intrinsic_data,
        intrinsic_mask=x_intrinsic_mask,
    )
)(z_init, solve_keys)

fig, ax = plt.subplots(figsize=(4.4, 4.2))
plot_context_samples(
    ax,
    combined_samples,
    f"Intrinsic on x = {TARGET_X:g}, classifier free guidance on label = {TARGET_LABEL}",
    reference=context_reference_coords,
)
ax.axvline(TARGET_X, ls="--", c="crimson", lw=1)
plt.tight_layout()
plt.show()

# Summary

**What we did.** Starting from a trained generative model, we steered its samples at sampling time through two independent channels, then combined them into a hand-written recipe:

- **Section 3, intrinsic guidance**: conditioned `joint_model` on its own `index` and `coordinates` modalities, using nothing but the model's own denoising ability.
- **Section 4, context conditioning**: conditioned `context_model` on a `label` that was never a modality at all, amplified through classifier-free guidance method.
- **Section 5**: combined both into one custom `GuidanceFn`, pinning `x` through intrinsic guidance while steering `label` through context, in the same sampling loop.

**The high-level takeaway**: `GuidanceFn` is a structural protocol, not a fixed set of recipes. `get_intrinsic_guidance_generator` and `get_classifier_free_guidance_generator` are two implementations of it, and Section 5 showed a third built from scratch. 